<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c37-header.png" alt="Nextia Learning · Recommendation and Personalisation: Show Each Person What Matters" width="100%">

# ALS and BPR with implicit

**[ALS and BPR with implicit](https://learning.nextia-ai.com/courses/recommendations/m05/als-and-bpr-with-implicit/)** · Recommendation and Personalisation: Show Each Person What Matters · Module 5, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** train ALS and BPR with the implicit library on the shop log, with confidence weights, and compare them by segment.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells (about 8 MB). No account and no key. |
| **You should know** | Customer and product vectors, from [Hidden tastes](https://learning.nextia-ai.com/courses/recommendations/m05/hidden-tastes/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, SciPy 1.18.1 and matplotlib 3.11.2 (implicit 0.7.3 or LightGBM 4.7.0 where the lesson needs it), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/recommendations/m05/als-and-bpr-with-implicit/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C37/M05-L02-als-and-bpr-with-implicit/als-and-bpr-with-implicit-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, scipy 1.18.1, implicit 0.7.3, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 scipy==1.18.1 matplotlib==3.11.2 implicit==0.7.3
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas', 'scipy', 'implicit']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 8 MB) into a folder `larkfield-recs` and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-recs` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop, and its shop log is **synthetic**: our code made the products, the customers and every view, basket and order, with a model of customer behaviour. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C37/data/dataset-card.md) explains how.

In [ ]:
# Setup: download the files into larkfield-recs/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C37/"
FILES = {
    "data/products.csv": "3da37a3809f6067da12eb5778b8abcf809545599db6959208c72a790861407ff",
    "data/customers.csv": "282446db7097aec2b30c27c574a6c7b13a8cffce8053fed212d80bf86cc1411c",
    "data/events.csv.gz": "3f46ddfff683586ab1548bc0e6756d85a439a1de941e6de3030edc383d06d1b3",
    "data/splits.json": "c58597803ced6fa787ff23c86d8ee6e9f0c1d7024c0ec109d2cca25caf6213e6",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-recs").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-recs and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-recs.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, SciPy's sparse matrices as `sp` and matplotlib. It defines the course's small helpers, the same ideas as the course project:

- `products()`, `customers()` and `events()` read the data. `events()` has one row per view, basket add or order, with the column `day` (days since 1 February 2025).
- `split(period)` gives the events before the period (`train`, to learn from) and, for each customer who ordered in the period, the products they ordered (`truth`). The periods are `"validation"`, `"test"` and `"launch"`.
- `recall_at_k(list, ordered, k)`, `ndcg_at_k(list, ordered, k)` and `coverage(lists, n_products)` measure suggestion lists.
- `show(ids)` prints products with their title and subcategory.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, the time split, the measures, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
START = pd.Timestamp("2025-02-01")

def products():
    """One row per product (2,994)."""
    df = pd.read_csv("data/products.csv", keep_default_na=False)
    for col in ("age_restricted", "sensitive", "in_stock"):
        df[col] = df[col].astype(str).eq("True")
    return df

def customers():
    """One row per customer (40,000)."""
    return pd.read_csv("data/customers.csv")

def events():
    """One row per event: view, basket or order. day = days since 1 February 2025."""
    ev = pd.read_csv("data/events.csv.gz", parse_dates=["timestamp"])
    ev["day"] = (ev["timestamp"] - START).dt.days
    return ev

def split(period, ev=None):
    """train = the events before the period; truth = {customer: set of products ordered in the period}."""
    ev = events() if ev is None else ev
    first, last = json.load(open("data/splits.json"))[period]
    a, b = (pd.Timestamp(first) - START).days, (pd.Timestamp(last) - START).days
    train = ev[ev["day"] < a]
    target = ev[(ev["day"] >= a) & (ev["day"] <= b) & (ev["event"] == "order")]
    truth = target.groupby("customer_id")["product_id"].agg(set).to_dict()
    return train, truth

def recall_at_k(rec, ordered, k=10):
    """Share of the ordered products that are in the first k suggestions."""
    return len(set(list(rec)[:k]) & set(ordered)) / len(ordered)

def ndcg_at_k(rec, ordered, k=10):
    """DCG of the list divided by the best possible DCG (discount log2(place + 1))."""
    gains = [1.0 if p in ordered else 0.0 for p in list(rec)[:k]]
    dcg = sum(g / np.log2(i + 2) for i, g in enumerate(gains))
    ideal = sum(1 / np.log2(i + 2) for i in range(min(len(ordered), k)))
    return dcg / ideal

def coverage(lists, n_products, k=10):
    """Share of the catalogue that appears in at least one list."""
    return len({p for rec in lists for p in list(rec)[:k]}) / n_products

def rounded(x, n=3):
    """x rounded to n places (None or ... unchanged: the task is not done yet)."""
    return x if x is None or x is ... else round(float(x), n)

def show(ids, n=10):
    """Print products: place, ID, title, subcategory."""
    global _PR
    if "_PR" not in globals():
        _PR = products().set_index("product_id")
    for i, p in enumerate(list(ids)[:n], 1):
        print(f"{i:2}. {p}  {_PR.loc[p, 'title'][:52]:<52}  {_PR.loc[p, 'subcategory']}")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Confidence: how sure are we?

An order says more than a view. An empty cell says almost nothing: the customer probably never saw the product. ALS for implicit feedback gives every cell a **confidence**. A filled cell gets **alpha × its weight**. An empty cell gets 1, with the target 0. Run the next cell: it adds up the weights of Ana's events.

In [ ]:
# A tiny example we made: Ana's events. Weights as in the course: view 1, basket 3, order 5.
tiny = pd.DataFrame({
    "product": ["garden bench", "garden bench", "garden bench", "parasol"],
    "event":   ["view",         "basket",       "order",        "view"],
})
weight = tiny.groupby("product")["event"].apply(lambda e: e.map({"view": 1, "basket": 3, "order": 5}).sum())
weight["dog bed"] = 0                          # no event: an empty cell
print(weight)

> **Predict.** With alpha = 2, how many times more does ALS care about Ana's garden bench than about the dog bed?

In [ ]:
alpha = 2.0
confidence_bench = ...   # the confidence of Ana's cell for the garden bench: alpha times its weight
print(confidence_bench)

In [ ]:
expect('confidence_bench', confidence_bench, 18.0)

> **Check.** The bench cell has confidence 2 × 9 = 18, and the empty dog-bed cell has confidence 1. ALS tries 18 times harder to give the bench a high score than to give the dog bed a low one.

## 2. The shop log as a sparse matrix

The next cell builds the matrix as the project does: one row per customer of `customers.csv`, one column per product of `products.csv`, the sum of the event weights before the test period. Only the filled cells are stored.

In [ ]:
import time
ev = events()
train, truth = split("test", ev)
pr, cu = products(), customers()
p_index = pd.Series(np.arange(len(pr)), index=pr["product_id"])
c_index = pd.Series(np.arange(len(cu)), index=cu["customer_id"])
w = train["event"].map({"view": 1.0, "basket": 3.0, "order": 5.0}).to_numpy()
M = sp.csr_matrix((w, (c_index[train["customer_id"]].to_numpy(), p_index[train["product_id"]].to_numpy())),
                  shape=(len(cu), len(pr)), dtype=np.float32)
M.sum_duplicates()
sparse_mb = (M.data.nbytes + M.indices.nbytes + M.indptr.nbytes) / 1e6
print(f"{M.shape[0]:,} customers x {M.shape[1]:,} products, {M.nnz:,} filled cells")
print(f"sparse: {sparse_mb:.1f} MB; dense it would be {M.shape[0] * M.shape[1] * 4 / 1e6:.0f} MB")

The next cell makes the lists as the project does. Products that are not on sale are never suggested, and a customer with no event before the period gets the trending list. Run it: it prints nothing.

In [ ]:
# The lists, as the project makes them: products on sale only; a customer with no event gets trending.
first_day = (pd.Timestamp(json.load(open("data/splits.json"))["test"][0]) - START).days
on_sale = ((pd.to_datetime(pr["added"]) - START).dt.days <= first_day).to_numpy()
recent = train[(train["event"] == "order") & (train["day"] >= first_day - 28)]
trend_count = recent.drop_duplicates(["customer_id", "product_id"]).groupby("product_id").size()
trend_count = trend_count.reindex(pr["product_id"], fill_value=0).to_numpy().astype(float)
trend_count[~on_sale] = 0
trending = [i for i in np.argsort(-trend_count, kind="stable") if trend_count[i] > 0][:10]
users = sorted(truth)
rows = c_index[users].to_numpy()
known = np.asarray(M.sum(axis=1)).ravel() > 0
ids = pr["product_id"].to_numpy()

def lists_from(model):
    scores = np.asarray(model.user_factors)[rows] @ np.asarray(model.item_factors).T
    scores[:, ~on_sale] = -np.inf
    top = np.argsort(-scores, axis=1, kind="stable")[:, :10]
    return [ids[t] if known[r] else ids[trending] for t, r in zip(top, rows)]

def mean_recall(lists):
    return float(np.mean([recall_at_k(l, truth[u]) for l, u in zip(lists, users)]))

## 3. ALS with the implicit library

These are the project's default settings: 64 numbers per vector, regularisation 0.05, alpha 2, 15 rounds, one thread. Run the cell (about 5 seconds).

In [ ]:
from implicit.als import AlternatingLeastSquares
als = AlternatingLeastSquares(factors=64, regularization=0.05, alpha=2.0, iterations=15, random_state=37,
                              num_threads=1)
t0 = time.perf_counter()
als.fit(M, show_progress=False)
print(f"ALS: fit {time.perf_counter() - t0:.1f} s")
als_lists = lists_from(als)
als_recall = mean_recall(als_lists)
print(f"recall@10 {als_recall:.3f}")

> **Check.** recall@10 is 0.142, the number that `recs evaluate als` prints. It is below trending (0.156): the course says this plainly.

## 4. Your turn: BPR

BPR learns the order of products instead of the cells: a product the customer touched should score above a random one they did not touch. Train it with the project's default settings (about 5 seconds).

> **Predict.** Will BPR's recall@10 be higher or lower than ALS's? Will it show more or fewer different products?

In [ ]:
from implicit.bpr import BayesianPersonalizedRanking
# Make a BPR model with factors=64, learning_rate=0.05, regularization=0.01, iterations=100, random_state=37,
# num_threads=1. Fit it on M, make the lists with lists_from(), and compute their mean recall@10.
bpr = ...
bpr_lists = ...
bpr_recall = ...
print(bpr_recall)

In [ ]:
expect('bpr_recall', rounded(bpr_recall, 3), 0.109, tolerance=0.002)

## 5. By segment

Run the next cell. It shows recall@10 for new, occasional and loyal customers, and the coverage: the share of the products on sale that appear in at least one list.

In [ ]:
before = train[train["event"] == "order"].groupby("customer_id")["session_id"].nunique()
n = before.reindex(users, fill_value=0).to_numpy()
segment = np.where(n == 0, "new", np.where(n <= 2, "occasional", "loyal"))
table = pd.DataFrame({"segment": segment,
                      "ALS": [recall_at_k(l, truth[u]) for l, u in zip(als_lists, users)]})
if bpr_lists is not ... and bpr_lists is not None:
    table["BPR"] = [recall_at_k(l, truth[u]) for l, u in zip(bpr_lists, users)]
print(table.groupby("segment").mean(numeric_only=True).reindex(["new", "occasional", "loyal"]).round(3))
print("coverage ALS:", round(coverage(als_lists, int(on_sale.sum())), 3))
if bpr_lists is not ... and bpr_lists is not None:
    print("coverage BPR:", round(coverage(bpr_lists, int(on_sale.sum())), 3))

> **Check.** BPR has recall@10 0.109, lower than ALS, and it is weakest for loyal customers. But it shows many more different products: its coverage is more than twice the coverage of ALS. The two methods learn different things from the same matrix.

In the project: `recs evaluate als --by segment` and `recs evaluate bpr --by segment` print the same numbers.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Tune without fooling yourself](https://learning.nextia-ai.com/courses/recommendations/m05/tune-without-fooling-yourself/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/recommendations/m05/als-and-bpr-with-implicit/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The Larkfield shop log is synthetic: our code made the products, the customers and everything they did, with a model of customer behaviour. No real customer is in this data.